In [5]:
import requests, jieba
import re
import os
from collections import Counter
from typing import List
import logging
jieba.setLogLevel(logging.ERROR) # 關閉 jieba 的預設日誌輸出

class SimpleRAG:
    def __init__(self, model_name="llama3:8b", base_url="http://localhost:11434"):
        self.model_name = model_name
        self.base_url = base_url
        self.chat_url = f"{base_url}/api/chat"
        self.documents = []
        self.processed_docs = []

    def _preprocess_jieba(self, text: str) -> List[str]:
        # 移除標點符號
        text = re.sub(r'[^\w\s\u4e00-\u9fff]', ' ', text)
        words = jieba.lcut(text)
        return [w.lower() for w in words if w.strip()]

    def _preprocess_text(self, text: str) -> List[str]:
        # 非 Jieba 模式：簡單的正則分詞
        text = re.sub(r'[^\w\s\u4e00-\u9fff]', ' ', text)
        words = []
        current_word = ""
        for char in text:
            if '\u4e00' <= char <= '\u9fff':
                if current_word:
                    words.append(current_word.lower())
                    current_word = ""
                words.append(char)
            elif char.isalnum():
                current_word += char
            else:
                if current_word:
                    words.append(current_word.lower())
                    current_word = ""
        if current_word:
            words.append(current_word.lower())
        return [word for word in words if word.strip()]
    
    def _calculate_tf_idf_similarity(self, query_words: List[str], doc_words: List[str]) -> float:
        if not query_words or not doc_words:
            return 0.0
        doc_word_set = set(doc_words)
        doc_word_count = Counter(doc_words)
        total_score = 0.0
        matched_words = 0
        for query_word in query_words:
            if query_word in doc_word_set:
                matched_words += 1
                tf = doc_word_count[query_word] / len(doc_words)
                word_weight = len(query_word) if len(query_word) > 1 else 1
                total_score += tf * word_weight
        if matched_words == 0:
            return 0.0
        match_ratio = matched_words / len(query_words)
        return total_score * match_ratio
    
    def _jaccard_similarity(self, query_words: List[str], doc_words: List[str]) -> float:
        set1 = set(query_words)
        set2 = set(doc_words)
        intersection = set1 & set2
        union = set1 | set2
        return len(intersection) / len(union) if union else 0.0
    
    def add_documents(self, documents: List[str], use_jieba: bool = True):
        self.documents = documents
        if use_jieba:
            self.processed_docs = [self._preprocess_jieba(doc) for doc in documents]
        else:
            self.processed_docs = [self._preprocess_text(doc) for doc in documents]
        
        print(f"✅ 已添加 {len(documents)} 個文檔到知識庫")
        
        # DEBUG: 輸出第一份文檔的前 30 個詞
        if self.processed_docs:
            DEB_K, DEB_L = 5,15
            print(f"🛠️ [DEBUG: Data Indexing] 前 5 頁斷詞範例 (每頁前 30 個詞):")
            for i, words in enumerate(self.processed_docs[:DEB_K]):
                page_title = self.documents[i].split('\n')[0]
                sample_words = words[:DEB_L]
                print(f"   {page_title} -> {sample_words}")
            print("-" * 50)
            
    def find_top_k_relevant_documents(self, query: str, method: str = "tfidf", k: int = 5, use_jieba: bool = True) -> tuple[list[str], list[float], list[int]]:
        if not self.documents: return [], [], []
        
        # 進行查詢端預處理
        query_words = self._preprocess_jieba(query) if use_jieba else self._preprocess_text(query)
        
        # DEBUG: 輸出 Query 斷詞結果
        print(f"🛠️ [DEBUG: Query Step] 問題斷詞結果:")
        print(f"   {query_words}")

        similarities = []
        for i, doc_words in enumerate(self.processed_docs):
            similarity = self._calculate_tf_idf_similarity(query_words, doc_words) if method == "tfidf" else self._jaccard_similarity(query_words, doc_words)
            similarities.append((i, similarity))
        
        top_k = sorted(similarities, key=lambda x: x[1], reverse=True)[:k]
        top_k_documents = [self.documents[i] for i, _ in top_k]
        top_k_similarities = [score for _, score in top_k]
        top_k_indices = [i for i, _ in top_k]
        return top_k_documents, top_k_similarities, top_k_indices
  
    def rag_top_k_query(self, query: str, method: str = "tfidf", k: int = 5, similarity_threshold: float = 0.01, useJieba: bool = True) -> str:
        # 獲取檢索結果
        top_k_docs, top_k_similarities, top_k_indices = self.find_top_k_relevant_documents(query, method, k=k, use_jieba=useJieba)
        
        # 輸出 檢索碎片 DEBUG 資訊
        print("\n--- [DEBUG: 檢索碎片資訊] ---")
        for rank, (idx, score) in enumerate(zip(top_k_indices, top_k_similarities), 1):
            page_info = self.documents[idx].split('\n')[0] # 取得標題行
            print(f"Rank {rank}: {page_info} | 相似度: {score:.4f}")
        print("----------------------------\n")

        filtered_docs_scores = [(doc, score) for doc, score in zip(top_k_docs, top_k_similarities) if score > similarity_threshold]
    
        if filtered_docs_scores:
            filtered_docs = [doc for doc, _ in filtered_docs_scores]
            filtered_scores = [score for _, score in filtered_docs_scores]
            references = "\n\n---\n\n".join(filtered_docs)
            prompt = f"""你是一個精準的資料分析助手。請根據提供的參考資料回答用戶問題。
                    【參考資料】
                    {references}
                    【問題】
                    {query}
                    【指令】
                    1. 若資料包含答案，請以繁體中文清楚、完整地列出。
                    2. 保持客觀，優先使用資料內的用語。
                    3. 如果資料中確實完全沒有提到相關資訊，請禮貌地回答「根據目前的資料庫，無法提供相關資訊」。"""    
            context_info = f"[使用RAG模式，採納 {len(filtered_docs)} 筆有效文檔]"
        else:
            prompt = f"問題：請以繁體中文回答，{query}\n注意：知識庫中無直接相關資料，請基於一般知識回答。"
            context_info = "[一般回答模式，相似度低於閾值]"
    
        print(f"📋 {context_info}")
        try:
            response = requests.post(self.chat_url, json={"model": self.model_name, "messages": [{"role": "user", "content": prompt}], "stream": False}, timeout=300)
            return response.json()["message"]["content"] if response.status_code == 200 else f"錯誤: {response.status_code}"
        except Exception as e:
            return f"請求錯誤: {str(e)}"

def main(input_path, model_name="qwen3:30b", use_jieba=True, top_k=3):
    print(f"🤖 Ollama {model_name} RAG 系統 (Jieba: {use_jieba}, Top-K: {top_k})")
    print("=" * 50)
    
    rag = SimpleRAG(model_name)
    documents = []

    if os.path.isdir(input_path):
        print(f"📂 偵測到資料夾模式：{input_path}")
        file_list = [f for f in os.listdir(input_path) if f.endswith('.txt')]
        file_list.sort(key=lambda f: int(re.search(r'\d+', f).group()) if re.search(r'\d+', f) else 0)
        
        for file_name in file_list:
            page_num_match = re.search(r'\d+', file_name)
            page_label = page_num_match.group() if page_num_match else file_name
            with open(os.path.join(input_path, file_name), "r", encoding="utf-8") as f:
                # 修改此處，移除 page_ 與 .txt
                documents.append(f"=== 第 {page_label} 頁 ===\n" + f.read().strip())
    else:
        print(f"📄 偵測到單一檔案模式：{input_path}")
        with open(input_path, "r", encoding="utf-8") as f:
            content = f.read()
            parts = content.split("=== 第")
            documents = ["=== 第" + p.strip() for p in parts[1:]]

    if not documents:
        print("❌ 無法讀取任何內容。")
        return

    print(f"✅ 已讀入 {len(documents)} 頁。")
    rag.add_documents(documents, use_jieba=use_jieba)
    
    print("\n🔍 請輸入查詢問題（輸入 quit 結束）")
    while True:
        query = input("❓ 問題: ").strip()
        if query.lower() == "quit": break
        answer = rag.rag_top_k_query(query, method="tfidf", k=top_k, useJieba=use_jieba)
        print(f"🤖 回答: {answer}\n" + "-" * 50)

if __name__ == "__main__":
    main(
        input_path="test_results",                            # pdf->img->md txt所在的目錄
        model_name="cwchang/llama-3-taiwan-8b-instruct:q8_0", # 處理 文字RAG的 語言模型
        use_jieba=True,                                       # 是否分詞提高TF-IDF準確率
        top_k=3                                               # 設定top k為3
    )

🤖 Ollama cwchang/llama-3-taiwan-8b-instruct:q8_0 RAG 系統 (Jieba: True, Top-K: 3)
📂 偵測到資料夾模式：test_results
✅ 已讀入 24 頁。
✅ 已添加 24 個文檔到知識庫
🛠️ [DEBUG: Data Indexing] 前 5 頁斷詞範例 (每頁前 30 個詞):
   === 第 1 頁 === -> ['第', '1', '頁', '113', '年', '9', '月', '19', '日本', '校', 'br', '招生', '委員會', '會議', '通過']
   === 第 2 頁 === -> ['第', '2', '頁']
   === 第 3 頁 === -> ['第', '3', '頁', '報名', '流程', '圖', '開始', '報名', '請', '詳閱', '招生', '簡章', '並依', '簡章', '日程']
   === 第 4 頁 === -> ['第', '4', '頁', '世新大學', '114444444444444444444444444444444']
   === 第 5 頁 === -> ['第', '5', '頁', '附錄', '一', '入學大學', '同等', '學力', '認定', '標準', '條', '文摘', '錄', '附錄', '二']
--------------------------------------------------

🔍 請輸入查詢問題（輸入 quit 結束）


❓ 問題:  quit
